<a href="https://colab.research.google.com/github/PradnyaKulkarni2005/NLP_Assignments/blob/main/NLP_Ass7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch torchvision torchaudio

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
training_data = [
    ("John lives in London".split(), ["PER","O","O","LOC"]),
    ("Mary works at Google".split(), ["PER","O","O","ORG"])
]

In [ ]:
# Create Word and Tag Dictionaries
word_to_ix = {}
tag_to_ix = {"PER":0, "O":1, "LOC":2, "ORG":3}

for sentence, tags in training_data:
    for word in sentence:
        if word not in word_to_ix:
            word_to_ix[word] = len(word_to_ix)
print(word_to_ix)

{'John': 0, 'lives': 1, 'in': 2, 'London': 3, 'Mary': 4, 'works': 5, 'at': 6, 'Google': 7}


In [ ]:
# Create Helper Function to Convert Words to Tensors
def prepare_sequence(seq, to_ix):
    idxs = [to_ix[w] for w in seq]
    return torch.tensor(idxs, dtype=torch.long)

In [ ]:
class LSTMTagger(nn.Module):

    def __init__(self, embedding_dim, hidden_dim, vocab_size, tagset_size):
        super(LSTMTagger, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.lstm = nn.LSTM(embedding_dim, hidden_dim)

        self.hidden2tag = nn.Linear(hidden_dim, tagset_size)

    def forward(self, sentence):

        embeds = self.embedding(sentence)

        lstm_out, _ = self.lstm(embeds.view(len(sentence), 1, -1))

        tag_space = self.hidden2tag(lstm_out.view(len(sentence), -1))

        tag_scores = torch.log_softmax(tag_space, dim=1)

        return tag_scores

In [ ]:
EMBEDDING_DIM = 6
HIDDEN_DIM = 6

model = LSTMTagger(EMBEDDING_DIM, HIDDEN_DIM, len(word_to_ix), len(tag_to_ix))
loss_function = nn.NLLLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

In [ ]:
# train the model
for epoch in range(100):

    for sentence, tags in training_data:

        model.zero_grad()

        sentence_in = prepare_sequence(sentence, word_to_ix)
        targets = prepare_sequence(tags, tag_to_ix)

        tag_scores = model(sentence_in)

        loss = loss_function(tag_scores, targets)

        loss.backward()
        optimizer.step()

In [ ]:
with torch.no_grad():
    inputs = prepare_sequence("John works at Google".split(), word_to_ix)
    tag_scores = model(inputs)
    print(tag_scores)

tensor([[-1.3565, -0.7146, -2.0021, -2.1368],
        [-2.2222, -0.1909, -3.4904, -3.3554],
        [-1.5899, -0.4191, -3.1706, -2.3386],
        [-1.2799, -1.5060, -1.5014, -1.2825]])
